# **Project Overview & Problem Statement**

* **The Core Problem:** Grid operators and households need accurate future energy forecasting to optimize distribution and reduce costs. However, time-series data is highly volatile, filled with missing data, and deeply influenced by human behavior and seasonal patterns.

* **The Project's Actual Scope:**  Before we can safely forecast the future, we must determine which modeling methodology handles this volatility best. Therefore, this project focuses on a comparative evaluation between classical statistical models (SARIMA) and modern automated forecasting frameworks (Meta Prophet) to identify the most robust predictive engine for household electricity consumption.**




#  **Dataset Documentation & Metadata**

##  Dataset Overview
* **Source:** Individual Household Electric Power Consumption Dataset
* **Temporal Horizon:** December 2006 – November 2010 (47 Months)
* **Total Volume:** 2,075,259 measurements (minute-by-minute granularity)
* **Data Integrity Note:** Approximately **1.25%** of rows contain missing values (represented as null spaces between separators). These require chronological time-based imputation.

---

##  Feature Architecture & Metadata

| Attribute Name | Data Type | Measurement Unit | Context / Description |
| :--- | :--- | :--- | :--- |
| **`Date`** | Object / Temporal | `dd/mm/yyyy` | Calendar date of recording. |
| **`Time`** | Object / Temporal | `hh:mm:ss` | Timestamps of minute-averaged logs. |
| **`Global_active_power`** | Numeric | Kilowatt (kW) | Household global minute-averaged active power. |
| **`Global_reactive_power`**| Numeric | Kilowatt (kW) | Household global minute-averaged reactive power. |
| **`Voltage`** | Numeric | Volt (V) | Minute-averaged voltage values. |
| **`Global_intensity`** | Numeric | Ampere (A) | Minute-averaged current intensity. |
| **`Sub_metering_1`** | Numeric | Watt-hour (Wh) | **Zone 1 (Kitchen):** Dishwasher, oven, microwave (excluding gas appliances). |
| **`Sub_metering_2`** | Numeric | Watt-hour (Wh) | **Zone 2 (Laundry Room):** Washing machine, tumble-dryer, refrigerator, lighting. |
| **`Sub_metering_3`** | Numeric | Watt-hour (Wh) | **Zone 3 (Climate Control):** Electric water-heater, air-conditioning units. |

---

##  Derived Feature Logic
The remaining unmetered active energy consumed every minute (in Watt-hours) by miscellaneous household equipment is mathematically expressed as:

$$\text{Active Energy Remainder} = \left( \frac{\text{Global\_active\_power} \times 1000}{60} \right) - \text{Sub\_metering\_1} - \text{Sub\_metering\_2} - \text{Sub\_metering\_3}$$

# **Phase 1: Environment Setup & Data Ingestion**

# Data Uploading & Ingestion

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
zip_ref = zipfile.ZipFile('household_power_consumption.txt.zip', 'r')
zip_ref.extractall

# Core Library Initialization
* **Action:** Importing primary data manipulation and exploratory visualization modules.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
os.listdir()

In [ ]:
df = pd.read_csv('household_power_consumption.txt.zip',sep = ';',low_memory=False)
df

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print(df.columns)

In [ ]:
print(df.iloc[0])

In [ ]:
df.iloc[10:30]

In [ ]:
print(f"the shape of the given dataset is{df.shape}")

## **Phase 2: Data Preprocessing & Structural Cleaning**

# Schema Enforcement & Data Type Realignment
 * **Action:** Parsing high-volume records and coercing anomalous string characters (like `?`) into numerical data types.

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

*because:*
*   understand structure
*   detect datatype
*   inspect possiable issues

In [ ]:
numeric_columns = [
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3'
]

for col in numeric_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors='coerce'
    )

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H:%M:%S'
)

* faster processing
* avoids format confusion
* professional practice for time-series projects

In [ ]:
df.head(10)

# DROPPING UNWANTED COLUMNS

In [ ]:
df.drop(['Date','Time'],axis=1,inplace= True)

In [ ]:
df.head()

# SETTING THE NEW CREATED COLUMN AS INDEX

In [ ]:
df.set_index('Datetime',inplace=True)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

#  Missing Value Imputation
* **Action:** Executing time-based linear interpolation (`method='time'`) to cleanly bridge data gaps (~1.25%) without losing temporal consistency.

In [ ]:
df.isnull().sum()

In [ ]:


df.isnull().sum().plot(kind='bar')

plt.title("Missing Values Before Handling")
plt.xlabel("Columns")
plt.ylabel("Number of Missing Values")

plt.show()

# Bar Plot Interpretation

The bar plot shows:

each column has around 25,979 missing values
all affected columns have nearly equal missing counts

This happened because missing rows affected multiple measurements at the same timestamps.

In [ ]:


sns.heatmap(df.isnull())

plt.show()

## Heatmap Interpretation

In the heatmap:

dark/black area means data is present (False)
light horizontal lines means missing values (True / NaN)

So those white horizontal lines indicate:

* missing readings at specific timestamps

Since this is time-series data, the heatmap helps identify:

* where gaps occurred over time
* whether missing values are continuous or scattered




## Why color bar shows 0 and 1

Internally:

False = 0
True = 1

because:

df.isnull()

creates boolean values.

So:

0 → not missing
1 → missing

The heatmap converts them into colors.

In [ ]:
df.interpolate(method= 'time',inplace=True)

### EXPLAINATION
method='time' was used because the dataset is a time-series dataset with a datetime index, and this method estimates missing values based on time intervals, preserving temporal continuity in energy consumption patterns.

In [ ]:
df.isnull().sum()

#  Record duplication
* **Action:** Identifying and dropping over 142,582 duplicate rows (~7%) to prevent structural bias in model training.

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.shape